# DRAFT — 04c mouse stromal de novo annotation

**Presentation scaffold only.** Generated 2026-08-10 from the committed decision
record `/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/resources/annotations/mouse_stromal.csv` and locked vocabulary `/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/resources/annotation_vocabulary.yaml`. The map below is
frozen from the CSV; this notebook does not infer, edit, or write annotations.

- resolution: `leiden_scVI_3.0`
- decision CSV SHA-256: `84fe2ed614de86e00403067f343980d8fd6966647d2c0313b760b9aeed7f11d2`
- vocabulary SHA-256: `df2709ede865ad407aed85073f8957bab5c09c5be829c11ef4e10e0804566666`


## 1. Frozen configuration and annotation map

In [ ]:
SPECIES = 'mouse'
LINEAGE = 'stromal'
RES = 'leiden_scVI_3.0'
REPO = '/n/lab_storage/eddy_lab/users/nhilgert/xmens_final'
CSV_PATH = '/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/resources/annotations/mouse_stromal.csv'
VOCAB_PATH = '/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/resources/annotation_vocabulary.yaml'
CSV_SHA256 = '84fe2ed614de86e00403067f343980d8fd6966647d2c0313b760b9aeed7f11d2'
VOCAB_SHA256 = 'df2709ede865ad407aed85073f8957bab5c09c5be829c11ef4e10e0804566666'
ANNOTATION = {
    "0": "efib_proliferative",
    "1": "efib_subluminal",
    "10": "efib_middle_induced",
    "11": "efib_middle_induced",
    "12": "efib_middle_induced",
    "13": "decidual_proliferative",
    "14": "efib_middle_induced",
    "15": "efib_middle_induced",
    "16": "decidual_border",
    "17": "decidual_mature",
    "18": "decidual_proliferative",
    "19": "decidual_proliferative",
    "2": "efib_subluminal",
    "20": "decidual_border",
    "21": "efib_subluminal",
    "22": "efib_myometrial",
    "23": "decidual_proliferative",
    "24": "decidual_apoptotic",
    "25": "decidual_proliferative",
    "26": "decidual_proliferative",
    "27": "decidual_proliferative",
    "28": "decidual_proliferative",
    "29": "decidual_border",
    "3": "efib_middle",
    "30": "decidual_proliferative",
    "31": "decidual_mature",
    "32": "decidual_proliferative",
    "33": "decidual_proliferative",
    "34": "decidual_mature",
    "4": "efib_middle",
    "5": "efib_middle",
    "6": "efib_subluminal",
    "7": "efib_myometrial",
    "8": "efib_middle_induced",
    "9": "efib_middle_induced"
}
MARKERS = {
    "fibroblast": [
        "Pdgfra",
        "Col1a1",
        "Col3a1",
        "Dcn",
        "Lum"
    ],
    "decidual": [
        "Prl8a2",
        "Ptgs2",
        "Bmp2",
        "Wnt4",
        "Hand2"
    ],
    "prolif": [
        "Mki67",
        "Top2a"
    ],
    "contam": [
        "Cdh1",
        "Pax8",
        "Pecam1",
        "Ptprc",
        "Myh11",
        "Sox6"
    ]
}

## 2. Load the frozen lineage latent and verify complete cluster coverage

In [ ]:
import hashlib, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd, scanpy as sc

REPO = Path(REPO)
LATENT = REPO / "data/interim" / f"latent_{SPECIES}_{LINEAGE}.h5ad"
adata = sc.read_h5ad(LATENT)
assert RES in adata.obs, f"{RES} absent from {LATENT}"
clusters = set(adata.obs[RES].astype(str).unique())
assert clusters == set(ANNOTATION), (
    f"frozen map mismatch: missing={sorted(clusters-set(ANNOTATION))}, "
    f"extra={sorted(set(ANNOTATION)-clusters)}")
print(f"{adata.n_obs:,} cells x {adata.n_vars:,} genes; {len(clusters)} clusters")

## 3. Raw marker dotplot by frozen cluster

In [ ]:
plot_sets = {name: [gene for gene in genes if gene in adata.var_names]
             for name, genes in MARKERS.items()}
plot_sets = {name: genes for name, genes in plot_sets.items() if genes}
if plot_sets:
    sc.pl.dotplot(adata, plot_sets, groupby=RES, standard_scale="var",
                  figsize=(10, max(6, 0.22 * len(clusters))), show=True)
else:
    print("No configured marker genes are present.")

## 4. Apply the committed map in memory using the shared producer

In [ ]:
import sys
sys.path.insert(0, str(REPO / "workflow/scripts"))
from apply_annotation import (apply_map, build_vocabulary_rollup,
                              validate_canonical_annotations)
import yaml

vocab = yaml.safe_load(Path(VOCAB_PATH).read_text())
rollup = build_vocabulary_rollup(vocab)
summary = apply_map(adata, RES, ANNOTATION, rollup=rollup, output_mode="fine")
kept = adata[~adata.obs.excluded].copy()
validate_canonical_annotations(kept, vocab, context=f"{SPECIES}/{LINEAGE} draft")
kept.obs["display_label"] = kept.obs.cell_subtype.astype(str).where(
    kept.obs.cell_subtype.astype(str).ne(""), kept.obs.cell_type.astype(str))
print(summary.to_string())
print("\ncanonical cell types:")
print(kept.obs.cell_type.value_counts().to_string())
print("\ncanonical subtypes (nonempty):")
print(kept.obs.loc[kept.obs.cell_subtype.ne(""), "cell_subtype"].value_counts().to_string())

## 5. Verification dotplot ordered by the frozen annotation

In [ ]:
if plot_sets:
    sc.pl.dotplot(kept, plot_sets, groupby="display_label", standard_scale="var",
                  figsize=(10, max(6, 0.35 * kept.obs.display_label.nunique())), show=True)
sc.pl.embedding(kept, "X_umap_scVI", color=[RES, "display_label"],
                legend_loc="right margin", show=True)

## 6. Freeze check — assert the committed inputs are unchanged

In [ ]:
def _sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert _sha(CSV_PATH) == CSV_SHA256, "decision CSV changed; regenerate this draft"
assert _sha(VOCAB_PATH) == VOCAB_SHA256, "vocabulary changed; regenerate this draft"
now = pd.read_csv(CSV_PATH, dtype=str)
now_map = dict(zip(now.cluster.str.strip(), now.label.str.strip()))
assert now_map == ANNOTATION, "embedded map differs from committed CSV"
print("PASS: CSV and vocabulary hashes match; embedded map is exact.")